# 🏦 Kaggle Pipeline — Llama 3.2 3B Banking Intent

Notebook này chạy pipeline Kaggle-first:
1. Cài Unsloth + dependencies
2. Clone repo
3. Preprocess full data + Train 3B + Package artifacts
4. Nén output để tải về

In [ ]:
# === 1) Config ===
GITHUB_USERNAME = "tzin1401"
REPO_NAME = "banking-intent-unsloth"
BRANCH = "main"

# Pipeline / training knobs
TRAIN_CONFIG = "configs/train_3b_t4x2.yaml"
TRAIN_PER_CLASS = 999   # 999 = dùng toàn bộ data (~130/class)
TEST_PER_CLASS = 999    # 999 = dùng toàn bộ test (~40/class)
SAMPLE_SEED = 42

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print("Repo:", REPO_URL)
print("Config:", TRAIN_CONFIG)
print(f"Data: train={TRAIN_PER_CLASS}/class, test={TEST_PER_CLASS}/class")

In [ ]:
# === 2) Install Unsloth + dependencies ===
# Phải cài Unsloth trước khi clone repo vì requirements.txt không cover Kaggle
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton -q
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "transformers<=5.5.0" "datasets<4.4.0" "dill>=0.3.9" pyyaml scikit-learn -q
print("\n✅ Dependencies installed")

In [ ]:
# === 3) Clone or update repo ===
import os

os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_NAME"] = REPO_NAME

if not os.path.exists(REPO_NAME):
    !git clone "$REPO_URL"

%cd "$REPO_NAME"
!git fetch origin
!git checkout "$BRANCH"
!git pull origin "$BRANCH"
!git rev-parse --short HEAD

In [ ]:
# === 4) Set env vars + Preprocess ===
import os

os.environ["TRAIN_CONFIG"] = TRAIN_CONFIG
os.environ["TRAIN_PER_CLASS"] = str(TRAIN_PER_CLASS)
os.environ["TEST_PER_CLASS"] = str(TEST_PER_CLASS)
os.environ["SAMPLE_SEED"] = str(SAMPLE_SEED)

!python scripts/preprocess_data.py

In [ ]:
# === 5) Train ===
!python scripts/train.py

In [ ]:
# === 6) Package artifacts ===
!python scripts/package_artifacts.py

In [ ]:
# === 7) Preview packaged artifact ===
!cat artifacts/LATEST.txt
!ls -lah artifacts

latest = open("artifacts/LATEST.txt", "r", encoding="utf-8").read().strip()
print("\nLatest run:", latest)
!ls -lah "artifacts/$latest"

In [ ]:
# === 8) Nén output để tải về ===
!zip -r /kaggle/working/banking_intent_3b_outputs.zip outputs/ sample_data/ configs/ artifacts/
print("\n✅ File zip sẵn sàng tải về: banking_intent_3b_outputs.zip")
print("   → Vào Output tab của notebook để download.")

---
## Optional: Push artifact về GitHub

Cần tạo GitHub Personal Access Token (PAT) có quyền push repo.
Chạy 2 cell dưới đây nếu muốn đẩy artifact lên branch.

In [ ]:
# === 9) Set secret envs ===
# Khuyến nghị dùng Kaggle Secrets thay vì hard-code
import os

os.environ["GITHUB_TOKEN"] = "your_github_pat"
os.environ["GIT_USER_NAME"] = "your-name"
os.environ["GIT_USER_EMAIL"] = "your-email@example.com"
os.environ["TARGET_BRANCH"] = BRANCH

In [ ]:
# === 10) Push artifact ===
!bash scripts/kaggle_push_artifacts.sh